## Imports

In [1]:
!pip install minari
!pip install gymnasium
!pip install "mujoco>=3.1.1,<=3.1.6"
!pip install "gymnasium-robotics>=1.2.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.2/26.2 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.5/805.5 kB 60.1 MB/s eta 0:00:00


In [2]:
import os
import gymnasium as gym
import minari
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import random
from torch.distributions import Normal
from torch.optim.lr_scheduler import CosineAnnealingLR
import gymnasium_robotics

# IQL Algorithm

### Q-network

In [3]:
class Q_network(nn.Module):
  def __init__(self, obs_dim, act_dim, hidden_size=256):
    super().__init__()
    self.ffn1 = nn.Linear(obs_dim + act_dim, hidden_size)
    self.ffn2 = nn.Linear(hidden_size, hidden_size)
    self.ffn3 = nn.Linear(hidden_size, 1)
    self.relu = nn.ReLU()

  def forward(self, obs, act):
    x = torch.cat([act, obs], dim=-1)
    x = self.relu(self.ffn1(x))
    x = self.relu(self.ffn2(x))
    x = self.ffn3(x)
    return x

### Value network

In [4]:
class V_network(nn.Module):
  def __init__(self, obs_dim, hidden_size=256):
    super().__init__()
    self.ffn1 = nn.Linear(obs_dim, hidden_size)
    self.ffn2 = nn.Linear(hidden_size, hidden_size)
    self.ffn3 = nn.Linear(hidden_size, 1)
    self.relu = nn.ReLU()

  def forward(self, obs):
    x = self.relu(self.ffn1(obs))
    x = self.relu(self.ffn2(x))
    x = self.ffn3(x)
    return x

### Policy network

In [5]:
class policy(nn.Module):
  def __init__(self, obs_dim, act_dim, hidden_size=256, dropout=0.1):
    super().__init__()
    self.ffn1 = nn.Linear(obs_dim, hidden_size)
    self.ffn2 = nn.Linear(hidden_size, hidden_size)
    self.mu = nn.Linear(hidden_size, act_dim)
    self.log_std = nn.Parameter(torch.zeros(act_dim))
    self.dropout = nn.Dropout(dropout)
    self.relu = nn.ReLU()

  def forward(self, obs):
    x = self.dropout(self.relu(self.ffn1(obs)))
    x = self.dropout(self.relu(self.ffn2(x)))
    mu = self.mu(x)
    log_std = self.log_std.clamp(-5, 2)
    return mu, log_std

# Training Algorithm

In [6]:
class Dataset():
  def __init__(self, dataset):
    self.observations = torch.FloatTensor(dataset['observations'])
    self.actions = torch.FloatTensor(dataset['actions'])
    self.rewards = torch.FloatTensor(dataset['rewards']).unsqueeze(-1)
    self.next_observations = torch.FloatTensor(dataset['next_observations'])
    self.terminals = torch.FloatTensor(dataset['terminals']).unsqueeze(-1)
    self.size = len(self.observations)

  def sample(self, batch_size):
    rand_indices = torch.randint(0, self.size, (batch_size,))
    return dict(
        observations=self.observations[rand_indices],
        actions=self.actions[rand_indices],
        rewards=self.rewards[rand_indices],
        next_observations=self.next_observations[rand_indices],
        terminals=self.terminals[rand_indices]
    )

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [8]:
SAVE_DIR = '/content/drive/MyDrive/rl/iql/franka_kitchen'
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# Dataset keys and shapes
# 'observations': (N, obs_dim)
# 'actions': (N, act_dim)
# 'rewards': (N,)
# 'next_observations': (N, obs_dim)
# 'terminals': (N,)

In [16]:
# Contains demos that have mixed success (some only 2 out of 4 successes)
raw_dataset = minari.load_dataset('D4RL/kitchen/mixed-v2', download=True)

dataset_dict = {
    'observations': [],
    'actions': [],
    'rewards': [],
    'next_observations': [],
    'terminals': []
}


for i, episode in enumerate(raw_dataset):
    # Kitchen observations are dictionaries, extra the observation
    obs = episode.observations['observation']
    dataset_dict['observations'].append(obs[:-1])
    dataset_dict['actions'].append(episode.actions)
    dataset_dict['rewards'].append(episode.rewards)
    dataset_dict['next_observations'].append(obs[1:])
    dataset_dict['terminals'].append(episode.terminations)

for k in dataset_dict:
    dataset_dict[k] = np.concatenate(dataset_dict[k], axis=0)

In [10]:
# Franka Kitchen: 60-dim obs, 9-dim action
STATE_DIM  = 60
ACTION_DIM = 9

STATE_DIM=59, ACTION_DIM=9


In [11]:
class IQLTraining():
  # IQL paper hyperparams for Franka Kitchen/Adroit: beta=0.5, tau=0.7, dropout=0.1
  def __init__(self, lr=3e-4, beta=0.5, tau=0.7, offline_training_steps=int(1e6),
               target_update_rate=0.005, batch_size=256, gamma=0.99):
    self.offline_training_steps = offline_training_steps
    self.gamma = gamma
    self.beta = beta
    self.tau = tau
    self.alpha = lr
    self.target_update_rate = target_update_rate
    self.batch_size = batch_size
    self.curr_steps = 0
    self.eval_trajectories = 50

    self.q1 = None
    self.q2 = None
    self.v = None
    self.policy = None
    self.pol_optimizer = None
    self.v_optimizer = None
    self.q_optimizer = None
    self.q1_target = None
    self.q2_target = None
    self.obs_mean = 0
    self.obs_std = 0

  def train(self):
    dataset = Dataset(dataset_dict)
    state_dim  = STATE_DIM
    action_dim = ACTION_DIM

    self.eval_env = raw_dataset.recover_environment()

    self.q1 = Q_network(state_dim, action_dim).to(device)
    self.q2 = Q_network(state_dim, action_dim).to(device)
    self.v  = V_network(state_dim).to(device)
    self.policy   = policy(state_dim, action_dim).to(device)
    self.q1_target = Q_network(state_dim, action_dim).to(device)
    self.q2_target = Q_network(state_dim, action_dim).to(device)

    self.q1_target.load_state_dict(self.q1.state_dict())
    self.q2_target.load_state_dict(self.q2.state_dict())

    self.pol_optimizer = torch.optim.Adam(self.policy.parameters(), lr=self.alpha, betas=(0.9, 0.999))
    self.q_optimizer   = torch.optim.Adam(list(self.q1.parameters()) + list(self.q2.parameters()),
                                           lr=self.alpha, betas=(0.9, 0.999))
    self.v_optimizer   = torch.optim.Adam(self.v.parameters(), lr=self.alpha, betas=(0.9, 0.999))

    scheduler = CosineAnnealingLR(self.pol_optimizer, T_max=self.offline_training_steps, eta_min=1e-6)

    self.obs_mean = torch.tensor(dataset_dict['observations'].mean(0), device=device, dtype=torch.float32)
    self.obs_std  = torch.tensor(dataset_dict['observations'].std(0) + 1e-6, device=device, dtype=torch.float32)

    best_average = float('-inf')
    average_eval_rewards = []

    for i in range(self.offline_training_steps):
      batch = dataset.sample(self.batch_size)
      batch_s  = (batch['observations'].to(device) - self.obs_mean) / self.obs_std
      batch_a  = batch['actions'].to(device)
      batch_r  = batch['rewards'].to(device)
      batch_ns = (batch['next_observations'].to(device) - self.obs_mean) / self.obs_std
      batch_d  = batch['terminals'].to(device)

      with torch.no_grad():
        q = torch.min(self.q1_target(batch_s, batch_a), self.q2_target(batch_s, batch_a))

      mu, log_std = self.policy(batch_s)
      std = torch.exp(log_std)
      dist = Normal(mu, std)
      log_probs = dist.log_prob(batch_a).sum(-1, keepdim=True)

      v = self.v(batch_s)
      adv = q - v
      v_loss = self.asymmetric_l2_loss(adv)

      next_v  = self.v(batch_ns).detach()
      q1_loss = torch.mean((batch_r + self.gamma * (1 - batch_d) * next_v - self.q1(batch_s, batch_a)) ** 2)
      q2_loss = torch.mean((batch_r + self.gamma * (1 - batch_d) * next_v - self.q2(batch_s, batch_a)) ** 2)
      q_loss  = q1_loss + q2_loss

      exp_adv      = torch.exp(self.beta * adv.detach()).clamp(float('-inf'), 100)
      policy_loss  = -torch.mean(exp_adv * log_probs)

      self.v_optimizer.zero_grad()
      v_loss.backward()
      self.v_optimizer.step()

      self.q_optimizer.zero_grad()
      q_loss.backward()
      self.q_optimizer.step()

      self.pol_optimizer.zero_grad()
      policy_loss.backward()
      self.pol_optimizer.step()
      scheduler.step()

      for p, tp in zip(self.q1.parameters(), self.q1_target.parameters()):
        tp.data.copy_((1 - self.target_update_rate) * tp + self.target_update_rate * p)
      for p, tp in zip(self.q2.parameters(), self.q2_target.parameters()):
        tp.data.copy_((1 - self.target_update_rate) * tp + self.target_update_rate * p)

      if self.curr_steps % 20000 == 0:
        average_50 = self.evaluation()
        print('=================================')
        print(f'Step {self.curr_steps}')
        print('Average reward over 50 rollouts: {:.2f}'.format(average_50))
        average_eval_rewards.append(average_50)
        if average_50 > best_average:
          print('NEW BEST AVERAGE REWARD!')
          best_average = average_50

      if self.curr_steps % 50000 == 0:
        torch.save(self.policy.state_dict(),
                   os.path.join(SAVE_DIR, f'iql_kitchen_policy_{self.curr_steps}.pth'))

      self.curr_steps += 1

    return average_eval_rewards

  def evaluation(self):
    eval_rewards = []
    self.policy.eval()

    for _ in range(self.eval_trajectories):
      obs_dict, _ = self.eval_env.reset()
      state = obs_dict['observation'] if isinstance(obs_dict, dict) else obs_dict
      state = torch.tensor(state, dtype=torch.float32, device=device)
      state = (state - self.obs_mean) / self.obs_std
      done  = False
      total_reward = 0

      while not done:
        with torch.no_grad():
          mu, _ = self.policy(state)
          action = mu.cpu().numpy()

        next_obs, reward, terminated, truncated, _ = self.eval_env.step(action)
        done = terminated or truncated
        total_reward += reward

        next_state = next_obs['observation'] if isinstance(next_obs, dict) else next_obs
        state = torch.tensor(next_state, dtype=torch.float32, device=device)
        state = (state - self.obs_mean) / self.obs_std

      eval_rewards.append(total_reward)

    self.policy.train()
    return sum(eval_rewards) / len(eval_rewards)

  def asymmetric_l2_loss(self, u):
    return torch.mean(torch.abs(self.tau - (u < 0).float()) * u**2)

In [12]:
iql = IQLTraining()
average_eval_rewards = iql.train()

Step 0
Average reward over 50 rollouts: 0.00
NEW BEST AVERAGE REWARD!
Step 20000
Average reward over 50 rollouts: 24.76
NEW BEST AVERAGE REWARD!
Step 40000
Average reward over 50 rollouts: 55.52
NEW BEST AVERAGE REWARD!
Step 60000
Average reward over 50 rollouts: 230.76
NEW BEST AVERAGE REWARD!
Step 80000
Average reward over 50 rollouts: 338.50
NEW BEST AVERAGE REWARD!
Step 100000
Average reward over 50 rollouts: 318.32
Step 120000
Average reward over 50 rollouts: 716.90
NEW BEST AVERAGE REWARD!
Step 140000
Average reward over 50 rollouts: 437.86
Step 160000
Average reward over 50 rollouts: 308.40
Step 180000
Average reward over 50 rollouts: 623.04
Step 200000
Average reward over 50 rollouts: 543.50
Step 220000
Average reward over 50 rollouts: 524.80
Step 240000
Average reward over 50 rollouts: 136.40
Step 260000
Average reward over 50 rollouts: 44.18
Step 280000
Average reward over 50 rollouts: 684.28
Step 300000
Average reward over 50 rollouts: 761.64
NEW BEST AVERAGE REWARD!
Step 32